# 📊 W8: Google Sheets 실시간 KPI 연동 (v2)
**hexa-1 — Week 8** | ⏱️ 60분 | 🎯 KPI → Sheets 자동 입력

> 💡 API 없이도 CSV 저장 모드로 전체 실습 가능

## Step 0: 환경 설정

In [ ]:
!pip install -q pandas gspread google-auth
print('✅ 설치 완료')


## Step 1: 설정 입력 (✏️)

In [ ]:
import pandas as pd, os, io
from datetime import datetime

SHEETS_CONFIG = {
    'spreadsheet_name': '✏️ [스프레드시트 이름]',
    'sheet_name': 'KPI_Daily',
    'use_api': False,  # True: 실제 Sheets / False: CSV 저장 (기본)
}

import requests
url = 'https://raw.githubusercontent.com/Reasonofmoon/hexa-1/main/scripts/oee_sample_data.csv'
try:
    df = pd.read_csv(url, encoding='utf-8-sig')
    print(f'✅ 샘플 데이터 로드: {len(df)}행')
except Exception as e:
    print(f'GitHub 로드 실패: {e}')
    from google.colab import files
    up = files.upload()
    if not up:
        raise RuntimeError('데이터 필요: CSV를 업로드하거나 인터넷 연결을 확인하세요.')
    df = pd.read_csv(io.BytesIO(list(up.values())[0]), encoding='utf-8-sig')
df.head(3)


## Step 2: CSV 저장 또는 Sheets 업로드

In [ ]:
# ✅ __wrapped__ 버그 수정 — 재귀 대신 명시적 CSV 저장 함수 분리
def save_as_csv(df, config):
    out = f"kpi_{config['sheet_name']}_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
    df.to_csv(out, index=False, encoding='utf-8-sig')
    print(f'💾 CSV 저장 완료: {out}')
    return out

def write_to_sheets(df, config):
    if not config['use_api']:
        return save_as_csv(df, config)
    try:
        import gspread
        from google.colab import auth
        auth.authenticate_user()
        from google.auth import default
        creds, _ = default()
        gc = gspread.authorize(creds)
        sh = gc.open(config['spreadsheet_name'])
        ws = sh.worksheet(config['sheet_name'])
        ws.clear()
        ws.update([df.columns.tolist()] + df.values.tolist())
        print(f'✅ Google Sheets 업데이트 완료: {len(df)}행')
    except Exception as e:
        print(f'⚠️ Sheets 연동 실패: {e} → CSV 저장으로 대체')
        return save_as_csv(df, config)  # ✅ __wrapped__ 제거, 직접 호출

result = write_to_sheets(df, SHEETS_CONFIG)
print('완료!')


## Step 3: 주간 집계 & 다운로드

In [ ]:
date_col = next((c for c in df.columns if '날짜' in c), df.columns[0])
df[date_col] = pd.to_datetime(df[date_col])
df['주차'] = df[date_col].dt.isocalendar().week
weekly = df.select_dtypes(include='number').groupby(df['주차']).mean().round(2)
print('📅 주간 평균 KPI:')
print(weekly.to_string())
weekly.to_csv('weekly_kpi_summary.csv', encoding='utf-8-sig')
from google.colab import files
files.download('weekly_kpi_summary.csv')
print('✅ 주간 KPI 다운로드 완료')


---
## 🔥 확장 과제
1. `use_api: True`로 바꾸고 실제 Sheets 업로드
2. 매일 자동 실행: Google Apps Script 트리거 설정
3. `sheets_kpi_template.py` CLI로 스케줄 실행